# 01 Get the data
Downloads ParlGov (elections and cabinets, development version of 4 Nov 2023, https://parlgov.org/data-info/), the 23 national stock indices and the candidates for the external market factor (Yahoo Finance via `yfinance`, daily closes from `PRICES_START` to the day before `DOWNLOAD_DATE`). Each file is downloaded once into `data/raw/` and never again, so every notebook works on the same frozen data.

`download_log.csv` has one row per raw file, written when the file first appears: the day it was actually downloaded, its SHA-256 and the package versions of that download. On every later run the notebook recomputes each hash and stops if a file no longer matches its row (Yahoo may revise past values, and a raw file must never change). The external-factor file was downloaded after the others: it has the same vintage in its name and the same end date, and its own download day in the log. `index_manifest.csv` records the coverage of each series, the external factor included.

In [1]:
import sys
from pathlib import Path

# Folder of helpers.py: the notebook's own folder or, on Nuvolos (where the kernel starts elsewhere), found by search.
# The search accepts only a code/ folder of THIS project (helpers.py, analysis.py and ../data/crises.csv) and stops
# if it finds more than one, so another project's helpers.py can never be imported by mistake.
def find_code_dir():
    found = set()
    for root in [Path.cwd(), Path("/files"), Path.home()]:
        for p in [root / "helpers.py", *root.glob("**/code/helpers.py")]:
            d = p.parent
            if p.exists() and (d / "analysis.py").exists() and (d.parent / "data" / "crises.csv").exists():
                found.add(d.resolve())
        if found:
            break
    if len(found) != 1:
        raise FileNotFoundError(f"expected one code/ folder of this project, found {sorted(map(str, found))}")
    return found.pop()


sys.path.insert(0, str(find_code_dir()))

import hashlib
import platform
from datetime import date
from importlib.metadata import PackageNotFoundError, version

import pandas as pd

from helpers import (DOWNLOAD_DATE, EXTERNAL_CANDIDATES, PARLGOV_TABLES, PARLGOV_URL, PRICES_START, RAW, TICKERS,
                     load_external, load_prices)

In [2]:
def yahoo(tickers):
    import yfinance as yf          # needed only for a download: reruns on frozen files work without it
    return yf.download(tickers, start=PRICES_START, end=DOWNLOAD_DATE, auto_adjust=True,
                       progress=False)["Close"].rename_axis("date")


def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not installed"


def download(name, fetch, index):
    """Save fetch() as data/raw/<name>_<DOWNLOAD_DATE>.csv unless the file already exists. Returns (path, status)."""
    path = RAW / f"{name}_{DOWNLOAD_DATE}.csv"
    if path.exists():
        return path, "already in data/raw"
    fetch().to_csv(path, index=index)
    return path, "downloaded now"


RAW.mkdir(parents=True, exist_ok=True)
files = [download(f"parlgov_{t}", lambda t=t: pd.read_csv(PARLGOV_URL + t + ".csv"), False) for t in PARLGOV_TABLES]
files += [download("index_prices", lambda: yahoo(list(TICKERS.values())), True),
          download("external_index", lambda: yahoo(list(EXTERNAL_CANDIDATES)), True)]
for path, status in files:
    print(f"{path.name}: {status}")

external, ticker = load_external()
print("External factor:", EXTERNAL_CANDIDATES[ticker], f"({ticker}), from", external.index[0].date())

parlgov_view_election_2026-11-19.csv: already in data/raw
parlgov_view_cabinet_2026-11-19.csv: already in data/raw
index_prices_2026-11-19.csv: already in data/raw
external_index_2026-11-19.csv: downloaded now
External factor: MSCI World (^990100-USD-STRD), from 1995-01-03


In [3]:
# Download log: one row per raw file, added when the file first appears and never rewritten.
LOG = RAW.parent / "download_log.csv"
COLS = ["file", "download_date", "bytes", "rows", "sha256", "yfinance", "pandas", "python"]
log = pd.read_csv(LOG, dtype=str) if LOG.exists() else pd.DataFrame(columns=COLS)

rows, changed = [], []
for path, status in files:
    sha = hashlib.sha256(path.read_bytes()).hexdigest()
    old = log[log["file"] == path.name]
    if len(old):
        if old["sha256"].iloc[0] != sha:
            changed.append(path.name)
        continue
    rows.append({"file": path.name,
                 # the day the file was written: today for a new download, the file's date for a file already there
                 "download_date": (date.today() if status == "downloaded now"
                                   else date.fromtimestamp(path.stat().st_mtime)).isoformat(),
                 "bytes": path.stat().st_size, "rows": len(pd.read_csv(path)), "sha256": sha,
                 "yfinance": package_version("yfinance") if "parlgov" not in path.name else "",
                 "pandas": pd.__version__, "python": platform.python_version()})
assert not changed, f"raw files that no longer match download_log.csv: {changed}. Stop: a raw file must never change."
if rows:
    log = pd.concat([log, pd.DataFrame(rows, columns=COLS)], ignore_index=True)
    log.to_csv(LOG, index=False)
print(f"download_log.csv: {len(rows)} new row(s); every hash checked against the log")
log[["file", "download_date", "sha256"]]

download_log.csv: 1 new row(s); every hash checked against the log


,file,download_date,sha256
0,parlgov_view_election_2026-11-19.csv,2026-11-19,d129b05b3f69671bdcb6beba4312aa5c4bc00a34e71439...
1,parlgov_view_cabinet_2026-11-19.csv,2026-11-19,eb2dda61669a86fa8737ce70eb2b61713303c74d47ae65...
2,index_prices_2026-11-19.csv,2026-11-19,c53b2d934c5158c884975ddd3ab052e3d86b0612b7c495...
3,external_index_2026-11-19.csv,2026-11-24,ab319aae0e013d0f432c7168cda1b8c8bcec5033b37344...


In [4]:
# Coverage of every series. Return type where it is documented; "unverified" otherwise.
RETURN_TYPE = {"^OMXTGI": "gross (total return)", "^OMXVGI": "gross (total return)", "^NZ50": "gross (total return)",
               "TA35.TA": "gross (labelled GTR by TASE)", "^OMX": "price", "^OMXH25": "price", "PSI20.LS": "price",
               "^GSPC": "price", "^990100-USD-STRD": "price (MSCI standard, USD)"}
prices = load_prices()[list(TICKERS.values())]
series = {**{c: prices[t].dropna() for c, t in TICKERS.items()}, "EXTERNAL": external}
manifest = pd.DataFrame({"country": list(series),
                         "ticker": [*TICKERS.values(), ticker],
                         "first_date": [s.index[0].date() for s in series.values()],
                         "last_date": [s.index[-1].date() for s in series.values()],
                         "n_days": [len(s) for s in series.values()],
                         "return_type": [RETURN_TYPE.get(t, "unverified") for t in [*TICKERS.values(), ticker]]}
                        ).sort_values("first_date")
manifest.to_csv(RAW.parent / "index_manifest.csv", index=False)
manifest

,country,ticker,first_date,last_date,n_days,return_type
11,ISR,TA35.TA,1995-01-02,2026-11-18,6302,gross (labelled GTR by TASE)
1,AUT,^ATX,1995-01-02,2026-11-18,7927,unverified
14,NLD,^AEX,1995-01-02,2026-11-18,8138,unverified
5,DEU,^GDAXI,1995-01-02,2026-11-18,8082,unverified
6,ESP,^IBEX,1995-01-02,2026-11-18,8084,unverified
8,GBR,^FTSE,1995-01-03,2026-11-18,8054,unverified
7,FRA,^FCHI,1995-01-03,2026-11-18,8121,unverified
0,AUS,^AXJO,1995-01-03,2026-11-18,8060,unverified
3,CAN,^GSPTSE,1995-01-03,2026-11-18,8012,unverified
2,BEL,^BFX,1995-01-03,2026-11-18,8122,unverified
